# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the FAIRˆ² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and Python data tools.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
ds_metadata = dataset.metadata
print(f"{ds_metadata.name}: {ds_metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs from the Croissant metadata.

In [ ]:
# List record sets with their @id and contents
record_sets = dataset.metadata.record_sets
if record_sets is None or len(record_sets) == 0:
    # If dataset.record_sets is empty, try to guess record sets by parsing download objects
    # or data schemas
    print("No record sets defined in metadata. Attempting to infer record sets from schema...")
    # Try via dataset.record_set_ids()
    record_set_ids = dataset.record_set_ids()
    if not record_set_ids:
        raise ValueError("No record sets found in the metadata.")
    record_sets = record_set_ids
    print("Record Set @ids:")
    for rid in record_sets:
        print(f"- {rid}")
else:
    print("Record sets found in the metadata:")
    for rs in record_sets:
        print(f"@id: {rs['@id']}")
        fields = rs.get('field', [])
        # fields may be dict or list
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for f in fields:
            if isinstance(f, dict):
                print(f"    - @id: {f.get('@id', 'unknown')} | Name: {f.get('name', 'N/A')}")
            else:
                print(f"    - @id: {f}")

## 3. Data Extraction
Load data from record sets into DataFrames for analysis, using `@id` references.

In [ ]:
# Extract data from each record set using their @ids
record_set_ids = dataset.record_set_ids()
print(f"Available record set @ids: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for record set @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame with {len(df)} rows and columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# Pick the main record set with most columns and show first rows
main_rs = max(dataframes, key=lambda k: len(dataframes[k].columns))
print(f"\nMain record set selected: {main_rs}")
print("Columns:", dataframes[main_rs].columns.tolist())
dataframes[main_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter, normalize, and group using field `@id`s.

We'll pick a numeric field and a group field for demonstration. All `@id` parameters are used for referencing.

In [ ]:
# Set up EDA
df = dataframes[main_rs]

# Show all field (column) @ids to help select
print("Available field @ids in main record set:")
for i, col in enumerate(df.columns):
    print(f"  {i}: {col}")
# For demonstration, pick two columns:
# Assume, based on dataset description and examining the top, typical fields might be:
#   '@id': 'age'            -> numerical
#   '@id': 'sex'            -> grouping/categorical

# Try to select most likely candidates for numeric and group fields
from difflib import get_close_matches

# Helper to find likely column @id
def find_field_id(possible_names, columns):
    for pn in possible_names:
        match = get_close_matches(pn, columns, n=1, cutoff=0.7)
        if match:
            return match[0]
    return columns[0]  # fallback

numeric_candidates = ['age', 'Age', 'age_at_second_crc', 'interval', 'interval_months', 'interval_years']
group_candidates = ['sex', 'Sex', 'gender', 'Sex (M/F)', 'msi_status', 'anatomical_location']

numeric_field = find_field_id(numeric_candidates, df.columns)
group_field = find_field_id(group_candidates, df.columns)

print(f"\nSelected numeric field: {numeric_field}")
print(f"Selected group field: {group_field}")

# Try to filter numeric field > threshold (pick meaningful threshold)
threshold = df[numeric_field].dropna().astype(float).mean()
filtered_df = df[df[numeric_field].astype(float) > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df[[numeric_field, group_field]].head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by group_field, compute mean and count
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean','count'])
    print(f"\nGrouped data by {group_field} (showing mean/count):")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships (e.g., boxplot, histogram) using numeric and group fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna().astype(float), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot by group_field (if categorical)
if group_field in df.columns and df[group_field].nunique() < 10:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field], y=df[numeric_field].astype(float))
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load and explore the FAIRˆ² dataset using the Croissant schema and Python tools via the `mlcroissant` library. We:
- Loaded dataset metadata and record sets via their `@id` fields
- Reviewed fields available for analysis using `@id` references
- Loaded records into pandas DataFrames for further exploration
- Performed basic EDA including filtering, normalization, and grouping
- Visualized numeric field distributions and relationships to categorical groupings

**Tip:** You can use the identified `@id` fields to further customize your analysis or automate loading of other FAIR datasets using Croissant schemas.